## Grobid Reference Extraction

In [6]:
import requests

from bs4 import BeautifulSoup

In [7]:
GROBID_API_URL = 'http://localhost:8070/api/processReferences'
PDF_FILE = '/home/willenjoy/Downloads/nrn.2016.164.pdf'
XML_FILE = '/home/willenjoy/Downloads/nrn.2016.164-refs.tei.xml'

In [8]:
"""
TODO: retry

https://github.com/kermitt2/grobid_client_python/blob/master/grobid_client.py
"""

def extract_references(pdf_file):
    files = {
        'input': {
            pdf_file,
            open(pdf_file, 'rb'),
            'application/pdf',
            {'Expires': '0'}
        }
    }
    
    res, status = requests.post(
        GROBID_API_URL,
        files=files,
        headers={'Accept': 'application/xml'}
    )
    
    return res, status

In [9]:
# res, status = extract_references(PDF_FILE)

In [10]:
with open(XML_FILE, 'r') as refs_xml:
    soup = BeautifulSoup(refs_xml, 'xml')

In [62]:
refs = soup.select('biblStruct')

In [63]:
len(refs)

222

In [34]:
pubmed_ref_ids = list(analyzer.citations_graph.successors(REVIEW_ID))

In [49]:
pubmed_data = analyzer.df[analyzer.df['id'].isin(pubmed_ref_ids)]
pubmed_ref_search_index = {v.lower(): k for k, v in zip(pubmed_data['id'], pubmed_data['title'])}
pubmed_ref_titles = {k: v for k, v in zip(pubmed_data['id'], pubmed_data['title'])}

In [52]:
pubmed_ref_titles['20692351'].lower() in pubmed_ref_search_index

True

In [73]:
found = 0
num2pmid = {}

for i, ref in enumerate(refs):
    if ref.analytic and ref.analytic.title:
        ref_title = ref.analytic.title.text
        if ref_title.lower() in pubmed_refs:
            pmid = pubmed_refs[ref_title.lower()]
            print(i + 1, ref_title, pmid)
            num2pmid[i + 1] = pmid
            found += 1
        
print(f'Found {found} references')

4 Real-time support vector classification and feedback of multiple emotional brain states 20692351
5 Online decoding of object-based attention using real-time fMRI 24438492
7 Neurofeedback as a nonpharmacological treatment for adults with attention-deficit/hyperactivity disorder (ADHD): study protocol for a randomized controlled trial 25928870
10 Real-time fMRI neurofeedback: progress and challenges 23541800
11 Selective attention from voluntary control of neurons in prefrontal cortex 21617042
12 Volitional modulation of optically recorded calcium signals during neuroprosthetic learning 24728268
13 High-performance neuroprosthetic control by an individual with tetraplegia 23253623
14 Neuronal ensemble control of prosthetic devices by a human with tetraplegia 16838014
15 Restoring cortical control of functional movement in a human with quadriplegia 27074513
16 Effects of neural synchrony on surface EEG 23236202
18 Mind over chatter: plastic up-regulation of the fMRI salience network dir

In [74]:
num2pmid

{4: '20692351',
 5: '24438492',
 7: '25928870',
 10: '23541800',
 11: '21617042',
 12: '24728268',
 13: '23253623',
 14: '16838014',
 15: '27074513',
 16: '23236202',
 18: '23022326',
 19: '15889581',
 20: '12824763',
 21: '17626211',
 22: '25762907',
 23: '26348555',
 24: '17336094',
 25: '22458676',
 28: '21840399',
 30: '16791142',
 31: '16899397',
 32: '17133383',
 33: '25870552',
 34: '22021045',
 35: '20888628',
 39: '25519874',
 40: '15852014',
 45: '22388818',
 46: '23954030',
 47: '16792290',
 48: '17057705',
 51: '25796342',
 53: '20570245',
 54: '24231399',
 55: '20384819',
 56: '21903976',
 58: '23536382',
 59: '16791145',
 60: '19540746',
 61: '25566028',
 62: '27620975',
 63: '23719815',
 65: '26971473',
 66: '10404201',
 67: '17234689',
 69: '21964375',
 71: '24151462',
 74: '687682',
 75: '7932682',
 76: '3567234',
 79: '24705203',
 80: '26948894',
 81: '26500521',
 82: '20977537',
 84: '23966933',
 85: '26066840',
 86: '15978025',
 90: '25673851',
 91: '23583615',
 92:

In [85]:
"""
FIXME:
 - how to resolve multiple occurrences in text? (currently use first occurrence)
 - numbers shift in current enumerate implementation, use ref ids in XML (after 78 +1, after 120 +3)
"""

SECTION_CLUSTERING_RAW = {
    'Neural specificity and plasticity': {
        'The neural substrates of self-regulation': [3, 4, 6] + list(range(11, 44)),
        'Neural plasticity and specificity': list(range(44, 68)),
    },
    'Neural mechanisms of self-regulation': {
        'Factors influencing neurofeedback learning': list(range(70, 86)),
        'Neural substrates of self-regulation': list(range(86, 94)),
    },
    'Clinical applications of neurofeedback': {
        'Attention deficit hyperactivity disorder': list(range(94, 137)),
        'Rehabilitation in stroke': list(range(138, 156)),
    }
}

In [86]:
clusters = []

for section_refs in SECTION_CLUSTERING_RAW.values():
    for subsection, raw_refs in section_refs.items():
        cluster = []
        for ref in raw_refs:
            if ref in num2pmid:
                cluster.append(num2pmid[ref])
        clusters.append(cluster)

In [87]:
for i, cluster in enumerate(clusters):
    print(f'Cluster {i + 1}')
    print(analyzer.df[analyzer.df['id'].isin(cluster)][['id', 'title']])

Cluster 1
           id                                              title
36   23253623  High-performance neuroprosthetic control by an...
64   20888628  Reorganization of functional and effective con...
73   22458676  Volitional reduction of anterior cingulate cor...
77   23236202         Effects of neural synchrony on surface EEG
86   15889581  Increasing individual upper alpha power by neu...
144  17133383    Real-time fMRI using brain-state classification
168  12824763  Ecological validity of neurofeedback: modulati...
174  25762907  Improvement in precision grip force control wi...
221  27074513  Restoring cortical control of functional movem...
233  17626211  Direct instrumental conditioning of neural act...
279  16791142  Decoding mental states from brain activity in ...
302  26348555  Enhanced control of dorsolateral prefrontal co...
303  16838014  Neuronal ensemble control of prosthetic device...
380  16899397  Beyond mind-reading: multi-voxel pattern analy...
385  15852014  

In [114]:
SECTION_CLUSTERING = {}

for i, cluster in enumerate(clusters):
    for ref in cluster:
        SECTION_CLUSTERING[ref] = i

In [115]:
SECTION_CLUSTERING

{'20692351': 0,
 '21617042': 0,
 '24728268': 0,
 '23253623': 0,
 '16838014': 0,
 '27074513': 0,
 '23236202': 0,
 '23022326': 0,
 '15889581': 0,
 '12824763': 0,
 '17626211': 0,
 '25762907': 0,
 '26348555': 0,
 '17336094': 0,
 '22458676': 0,
 '21840399': 0,
 '16791142': 0,
 '16899397': 0,
 '17133383': 0,
 '25870552': 0,
 '22021045': 0,
 '20888628': 0,
 '25519874': 0,
 '15852014': 0,
 '22388818': 1,
 '23954030': 1,
 '16792290': 1,
 '17057705': 1,
 '25796342': 1,
 '20570245': 1,
 '24231399': 1,
 '20384819': 1,
 '21903976': 1,
 '23536382': 1,
 '16791145': 1,
 '19540746': 1,
 '25566028': 1,
 '27620975': 1,
 '23719815': 1,
 '26971473': 1,
 '10404201': 1,
 '17234689': 1,
 '24151462': 2,
 '687682': 2,
 '7932682': 2,
 '3567234': 2,
 '24705203': 2,
 '26948894': 2,
 '26500521': 2,
 '20977537': 2,
 '23966933': 2,
 '26066840': 2,
 '15978025': 3,
 '25673851': 3,
 '23583615': 3,
 '22732558': 3,
 '25566020': 3,
 '6487671': 4,
 '11449024': 4,
 '10454276': 4,
 '26748531': 4,
 '7786929': 4,
 '19712709': 4

In [116]:
APPLICATION_CLUSTERING = {}

for ref in clusters[4]:
    APPLICATION_CLUSTERING[ref] = 0
for ref in clusters[5]:
    APPLICATION_CLUSTERING[ref] = 1

In [117]:
APPLICATION_CLUSTERING

{'6487671': 0,
 '11449024': 0,
 '10454276': 0,
 '26748531': 0,
 '7786929': 0,
 '19712709': 0,
 '24399101': 0,
 '19207632': 0,
 '22877086': 0,
 '23665196': 0,
 '19715181': 0,
 '25220088': 0,
 '22617866': 0,
 '24321363': 0,
 '25870550': 0,
 '22608481': 0,
 '25461225': 0,
 '25329936': 0,
 '23518223': 0,
 '15039008': 0,
 '23264371': 0,
 '18762860': 0,
 '26706052': 0,
 '15827991': 0,
 '24103255': 0,
 '23575842': 0,
 '17919929': 0,
 '20538065': 0,
 '17670949': 0,
 '12738893': 1,
 '23532011': 1,
 '25712802': 1,
 '22645108': 1,
 '26671217': 1,
 '22401758': 1,
 '20303409': 1,
 '23565083': 1,
 '26219602': 1,
 '27071949': 1}